In [ ]:
# Imports
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import torch.optim as optim
from torch.utils.data import DataLoader

In [ ]:
# Constants and Configurations
EPS = 1e-8
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8
beta = .01
lambda_patch = 0.1
epochs = 50
grad_clip = 5.0

In [ ]:
# Load training datasets

# T1-weighted healthy cohort metadata
# includes:
# - path: path to cleaned .npy MRI file
# - age_at_session: subject age at scan
# - sex: biological sex (encoded numerically)
# - OASIS3_id: subject identifier
# - CDR-SB: clinical dementia rating sum of boxes score
t1w_healthy_clinical = pd.read_csv(
    "YOUR PATH"
)

# sample one scan per subject to avoid repeated subject bias in training set
train_df_t1w = (
    t1w_healthy_clinical
    .groupby("OASIS3_id", group_keys=False)
    .apply(lambda x: x.sample(n=1))
)

# T2-weighted healthy cohort metadata
# includes:
# - path: path to cleaned .npy MRI file
# - age_at_session: subject age at scan
# - sex: biological sex (encoded numerically)
# - OASIS3_id: subject identifier
# - CDR-SB: clinical dementia rating sum of boxes score
t2w_healthy_clinical = pd.read_csv(
    "YOUR PATH"
)

# sample one scan per subject to avoid repeated subject bias in training set
train_df_t2w = (
    t2w_healthy_clinical
    .groupby("OASIS3_id", group_keys=False)
    .apply(lambda x: x.sample(n=1))
)

In [ ]:
class MRIDataset(torch.utils.data.Dataset):
    # custom dataset for loading MRI volumes and associated metadata

    def __init__(self, paths, ages, sexes, cdrs=None):
        self.paths = paths      # file paths to MRI volumes (.npy)
        self.ages = ages        # subject ages
        self.sexes = sexes      # subject sexes (encoded numerically); 0 for male and 1 for female
        self.cdrs = cdrs        # optional CDR-SB scores (labels)

    def __len__(self):
        # total number of samples
        return len(self.paths)  

    def __getitem__(self, idx):
        # load MRI volume
        volume = np.load(self.paths[idx]).astype(np.float32)

        # normalize volume to [0, 1]
        vmin, vmax = volume.min(), volume.max()
        if vmax - vmin > EPS:
            volume = (volume - vmin) / (vmax - vmin)
        else:
            volume = np.zeros_like(volume) 

        volume = torch.tensor(volume, dtype=torch.float32)

        # load metadata
        age = torch.tensor(self.ages[idx], dtype=torch.float32)
        sex = torch.tensor(self.sexes[idx], dtype=torch.float32)

        # include label if available
        if self.cdrs is not None:
            cdr = torch.tensor(self.cdrs[idx], dtype=torch.float32)
            return volume, age, sex, cdr

        return volume, age, sex

In [ ]:
# Data Preparation

# training datasets/loaders (no CDR-SB labels) for both T1w and T2w MRIs

train_dataset_t1w = MRIDataset(
    paths=train_df_t1w["fixed_path"].to_list(),
    ages=train_df_t1w["age_at_session"].to_list(),
    sexes=train_df_t1w["sex"].to_list()
)

train_loader_t1w = DataLoader(
    train_dataset_t1w,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

train_dataset_t2w = MRIDataset(
    paths=train_df_t2w["fixed_path"].to_list(),
    ages=train_df_t2w["age_at_session"].to_list(),
    sexes=train_df_t2w["sex"].to_list()
)

train_loader_t2w = DataLoader(
    train_dataset_t2w,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

In [ ]:
# Model

# Demographic Embedding
class DemographicEmbedding(nn.Module):
    """
    Encodes structured demographic variables (age, sex)
    into a learned embedding vector that conditions the model
    """

    def __init__(self, embed_dim=32):
        super().__init__()

        # simple MLP to project 2 demographic inputs into the embedding space
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, embed_dim),
            nn.ReLU()
        )

    def forward(self, age, sex):
        """
        age: tensor [B]
        sex: tensor [B]

        returns:
            embedding [B, embed_dim]
        """
        # stack age and sex into a single feature vector per subject
        demo = torch.stack([age, sex], dim=1)
        return self.net(demo)


# Bayesian Skip Connection
class BayesianSkip(nn.Module):
    """
    Stochastic skip connection that injects uncertainty using a learned mean and variance.
    """

    def __init__(self, channels, demo_dim=32):
        super().__init__()

        # predict feature-wise mean
        self.mu = nn.Conv3d(channels + demo_dim, channels, 1)

        # predict feature-wise log-variance (uncertainty)
        self.logvar = nn.Conv3d(channels + demo_dim, channels, 1)

    def forward(self, x, demo):
        """
        x: feature map [B, C, D, H, W]
        demo: demographic embedding [B, demo_dim]
        """

        B, C, D, H, W = x.shape

        # expand demographic embedding to match 3D spatial dimensions
        demo = demo.unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
        demo = demo.expand(B, demo.shape[1], D, H, W)

        x = torch.cat([x, demo], dim=1)

        # predict distribution parameters
        mu = self.mu(x)
        logvar = torch.clamp(self.logvar(x), -4, 4)  

        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)

        out = mu + eps * std

        return torch.nan_to_num(out, nan=0.0)


# ============================================================
# Encoder (3D CNN VAE encoder)
# ============================================================
class Encoder(nn.Module):
    """
    3D convolutional encoder that compresses MRI volumes
    into a latent Gaussian representation (mu, logvar).
    """

    def __init__(self, latent_dim):
        super().__init__()

        # progressive downsampling through strided convolutions
        self.conv1 = nn.Conv3d(1, 32, 3, 2, 1)
        self.conv2 = nn.Conv3d(32, 64, 3, 2, 1)
        self.conv3 = nn.Conv3d(64, 128, 3, 2, 1)
        self.conv4 = nn.Conv3d(128, 256, 3, 2, 1)

        # normalization stabilizes training for deep 3D CNNs
        self.bn1 = nn.BatchNorm3d(32)
        self.bn2 = nn.BatchNorm3d(64)
        self.bn3 = nn.BatchNorm3d(128)
        self.bn4 = nn.BatchNorm3d(256)

        # flattened feature size after the convolutional stack
        self.flatten_dim = 256 * 10 * 12 * 10

        # latent distribution parameters
        self.mu = nn.Linear(self.flatten_dim, latent_dim)
        self.logvar = nn.Linear(self.flatten_dim, latent_dim)

    def forward(self, x):
        """
        x: MRI volume [B, 1, D, H, W]
        """

        s1 = F.relu(self.bn1(self.conv1(x)))
        s2 = F.relu(self.bn2(self.conv2(s1)))
        s3 = F.relu(self.bn3(self.conv3(s2)))
        s4 = F.relu(self.bn4(self.conv4(s3)))

        # flatten bottleneck feature map
        flat = s4.view(s4.size(0), -1)

        # Gaussian latent parameters
        mu = self.mu(flat)
        logvar = torch.clamp(self.logvar(flat), -4, 4)

        return mu, logvar, (s1, s2, s3), s4


# ============================================================
# Decoder (3D CNN VAE decoder)
# ============================================================
class Decoder(nn.Module):
    """
    Reconstructs MRI volumes from latent representation
    using transposed convolutions + stochastic skip connections.
    """

    def __init__(self, latent_dim, demo_dim=32):
        super().__init__()

        # project latent vector back into 3D feature space
        self.fc = nn.Linear(latent_dim, 256 * 10 * 12 * 10)

        # stochastic skip connections conditioned on demographics
        self.bskip1 = BayesianSkip(128, demo_dim)
        self.bskip2 = BayesianSkip(64, demo_dim)
        self.bskip3 = BayesianSkip(32, demo_dim)

        # upsampling path (mirror of encoder)
        self.deconv1 = nn.ConvTranspose3d(256, 128, 3, 2, 1, 1)
        self.deconv2 = nn.ConvTranspose3d(128, 64, 3, 2, 1, 1)
        self.deconv3 = nn.ConvTranspose3d(64, 32, 3, 2, 1, 1)
        self.deconv4 = nn.ConvTranspose3d(32, 1, 3, 2, 1, 1)

        # normalization layers for stability
        self.bn1 = nn.BatchNorm3d(128)
        self.bn2 = nn.BatchNorm3d(64)
        self.bn3 = nn.BatchNorm3d(32)

    def forward(self, z, skips, demo):
        """
        z: latent vector [B, latent_dim]
        skips: encoder feature maps (s1, s2, s3)
        demo: demographic embedding [B, demo_dim]
        """

        s1, s2, s3 = skips

        # expand latent vector into 3D feature volume
        x = self.fc(z)
        x = x.view(-1, 256, 10, 12, 10)

        # progressive decoding with skip connections
        x = F.relu(self.bn1(self.deconv1(x))) + self.bskip1(s3, demo)
        x = F.relu(self.bn2(self.deconv2(x))) + self.bskip2(s2, demo)
        x = F.relu(self.bn3(self.deconv3(x))) + self.bskip3(s1, demo)

        # final reconstruction (voxel intensities in [0,1])
        return torch.sigmoid(self.deconv4(x))


# Variational Autoencoder (VAE)
class VAE(nn.Module):
    """
    Full variational autoencoder combining:
    - 3D MRI encoder/decoder
    - demographic conditioning
    - stochastic latent sampling
    """

    def __init__(self, latent_dim=256):
        super().__init__()

        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)
        self.demo_embed = DemographicEmbedding()

    def reparameterize(self, mu, logvar):
        """
        Samples latent vector using reparameterization trick
        while maintaining differentiability.
        """

        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)

        z = mu + eps * std
        z = torch.nan_to_num(z, nan=0.0)

        z = z / (z.norm(dim=1, keepdim=True) + 1e-8)

        return z

    def forward(self, x, age, sex):
        """
        x: MRI volume [B, 1, D, H, W]
        age: [B]
        sex: [B]
        """

        # encode input volume
        mu, logvar, skips, bottleneck = self.encoder(x)

        # sample latent representation
        z = self.reparameterize(mu, logvar)

        # embed demographics
        demo = self.demo_embed(age, sex)

        # reconstruct MRI conditioned on latent + demographics
        recon = self.decoder(z, skips, demo)

        return recon, mu, logvar, z, skips, bottleneck


# Loss functions

def patch_contrastive_loss(feat):
    """
    Encourages local feature consistency by comparing
    patch-wise similarity structure within feature maps.
    """

    feat = F.adaptive_avg_pool3d(feat, (5, 6, 5))

    B, C, D, H, W = feat.shape

    # reshape into patch descriptors
    feat = feat.view(B, C, -1)

    feat = F.normalize(feat, dim=1)

    # compute self-similarity matrix between patches
    sim = torch.bmm(feat.transpose(1, 2), feat)

    # identity matrix removes self-similarity bias
    eye = torch.eye(sim.size(-1), device=feat.device)

    return ((sim - eye) ** 2).mean()


def reconstruction_loss(x, recon):
    """
    Measures voxel-level reconstruction error (L1 loss).
    """
    return F.l1_loss(recon, x)


def kl_loss_func(mu, logvar):
    """
    KL divergence between learned latent distribution
    and standard normal prior.
    """
    return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

In [ ]:
def train_vae(
    model,
    dataloader,
    optimizer,
    device,
    epochs=50,
    beta=0.01,
    lambda_patch=0.1,
    grad_clip=5.0
):
    """
    Trains the VAE model and returns per-epoch loss history.

    Args:
        model: VAE model
        dataloader: training dataloader
        optimizer: optimizer for model parameters
        device: torch device (cpu/cuda)
        epochs: number of training epochs
        beta: weight for KL divergence loss
        lambda_patch: weight for contrastive loss
        grad_clip: gradient clipping threshold
        
    Returns:
        list of average losses per epoch
    """

    model.train()
    epoch_losses = []

    print("\nTRAINING VAE\n")

    for epoch in range(epochs):
        total_loss = 0.0

        for mri, age, sex in dataloader:

            mri = mri.to(device)
            age = age.to(device).view(-1)
            sex = sex.to(device).view(-1)

            # forward pass
            recon, mu, logvar, z, skips, bottleneck = model(mri, age, sex)

            # losses
            recon_loss = reconstruction_loss(mri, recon)
            kl = kl_loss_func(mu, logvar)
            contrastive = patch_contrastive_loss(bottleneck)

            loss = recon_loss + beta * kl + lambda_patch * contrastive

            # optimization step
            optimizer.zero_grad()
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        epoch_losses.append(avg_loss)

        print(f"Epoch {epoch + 1}/{epochs} | Loss: {avg_loss:.4f}")

    return epoch_losses

In [ ]:
# Model initialization

# T1-weighted 
model_t1w = VAE(latent_dim=64).to(device)
optimizer_t1w = torch.optim.Adam(model_t1w.parameters(), lr=1e-4)

# T2-weighted 
model_t2w = VAE(latent_dim=64).to(device)
optimizer_t2w = torch.optim.Adam(model_t2w.parameters(), lr=1e-4)

In [ ]:
# Training runs

# train VAE on T1-weighted data
losses_t1w = train_vae(model_t1w, train_loader_t1w, optimizer_t1w, device, epochs, beta, lambda_patch, grad_clip)

# train VAE on T2-weighted data
losses_t2w = train_vae(model_t2w, train_loader_t2w, optimizer_t2w, device, epochs, beta, lambda_patch, grad_clip)

In [ ]:
# Save Models\

# save T1-weighted model
torch.save(model_t1w.state_dict(), "model_t1w.pt")

# save T2-weighted model
torch.save(model_t2w.state_dict(), "model_t2w.pt")